In [30]:
import argparse
import os
import sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

In [50]:
from utils import (
    preprocess_contigs,
    KMediod,
    align_labels_via_linear_sum_assignemt,
    compute_eval_metrics,
)

#from threshold_eisuke import compute_class_center_medium_similarity

In [58]:
def compute_class_center_medium_similarity(
    embeddings, labels, sample_rate=0.2, sample_type="class_sample"
):
    """
    Compute similarity metrics by sampling embeddings within each class or among classes.

    Args:
        embeddings (np.ndarray): Embeddings of data points.
        labels (np.ndarray): Class labels for the embeddings.
        sample_rate (float): Proportion of data to sample from each class or number of classes.
        sample_type (str): 'class_sample' for sampling within each class,
                           'classwise_sample' for sampling whole classes.

    Returns:
        tuple: List of percentile values representing similarity distributions,
               List of indices of sampled data points,
               List of labels of sampled classes if sample_type is 'classwise_sample'.
    """
    unique_labels = np.unique(labels)
    all_similarities = []
    sampled_indices_list = []  # To store indices of sampled points

    # Sampled labels only if sample_type is 'classwise_sample'
    sampled_labels = None
    if sample_type == "classwise_sample":
        sampled_labels = np.random.choice(
            unique_labels, int(len(unique_labels) * sample_rate), replace=False
        )

    # Process each class based on the specified sampling strategy
    for label in unique_labels:
        class_indices = np.where(labels == label)[0]
        class_embeddings = embeddings[class_indices]

        # Skip classes with fewer than 10 samples
        if len(class_embeddings) <= 10:
            continue

        if sample_type == "class_sample":
            # Sample a percentage of data points within the class
            n_sample = max(1, int(len(class_embeddings) * sample_rate))
            sampled_indices = np.random.choice(
                len(class_embeddings), n_sample, replace=False
            )
            sampled_embeddings = class_embeddings[sampled_indices]
            sampled_original_indices = class_indices[sampled_indices]

        elif sample_type == "classwise_sample":
            # Check if the current class is in the sampled labels
            if label not in sampled_labels:
                continue
            sampled_embeddings = class_embeddings
            sampled_original_indices = class_indices  # Use all indices in this class

        # Calculate the mean and similarities for the sampled embeddings
        mean_embedding = np.mean(sampled_embeddings, axis=0)
        print(mean_embedding)
        similarities = np.dot(sampled_embeddings, mean_embedding)

        # Collect all similarities and sampled indices for percentile calculation
        all_similarities.extend(similarities)
        sampled_indices_list.extend(
            sampled_original_indices
        )  # Add original indices to the list

    # Compute and return percentile values for the aggregated similarities
    all_similarities = np.array(all_similarities)
    all_similarities.sort()
    
    percentile_values = [
        np.percentile(all_similarities, p) for p in [10, 20, 30, 40, 50, 60, 70, 80, 90]
    ]

    print(percentile_values)
    return percentile_values, sampled_indices_list

In [41]:
file_path = "contigs.csv"
contigs = pd.read_csv(file_path)
contigs = contigs.iloc[:50000, :]
print(contigs.shape)

labels = contigs.iloc[:, 2]
print(labels[0])

(50000, 3)
Acidaminococcus_D21_uid55871


In [34]:
label2id = {l: i for i, l in enumerate(set(labels))}
labels_bin = np.array([label2id[l] for l in labels])
print(labels_bin[0])

28


In [94]:
hyena = np.load("hyenadna.npy")[:50000]

In [101]:
dnaberts = []  
keep_going = True
with open('dnaberts.npy', 'rb') as f:  
    while keep_going:
        try:
            a = np.load(f)
            dnaberts.append(a) 
        except:
            keep_going = False

dnaberts = np.array(dnaberts)
dnaberts = np.squeeze(dnaberts, axis=1)
dnaberts = dnaberts[:50000]

In [100]:
dnaberts.shape

(50497, 768)

In [102]:
percentile_values, sampled_indices_list = compute_class_center_medium_similarity(
        dnaberts, labels
    )

[-1.14404242e-02  2.35467032e-01  1.18209474e-01 -1.26647770e-01
 -9.43247303e-02  1.01170182e-01  1.42558003e-02  1.06706910e-01
 -1.02562485e-02  4.37569134e-02 -7.13338256e-02 -9.19840485e-02
  1.20251924e-01 -1.14718534e-01  1.22858293e-01 -6.60784096e-02
 -1.01268671e-01  9.26736742e-02  2.85289194e-02 -1.13259312e-02
  2.04425082e-01 -2.27305684e-02  8.77017975e-02 -1.11634307e-01
  8.44203234e-02 -5.17272167e-02 -4.16081026e-02 -3.22339088e-02
  1.42381433e-02  1.07965786e-02 -2.31967077e-01 -2.04544254e-02
  9.53187793e-03 -9.45240855e-02  1.37737766e-01 -2.26154309e-02
 -7.64096761e-03  2.00390518e-02  5.17194830e-02  6.36108071e-02
  1.63293257e-02  1.24550000e-01  3.34412456e-02  4.08873484e-02
 -2.38576587e-02 -1.08939712e-03 -1.31690115e-01  3.59363332e-02
 -9.85422730e-02  1.20901085e-01 -9.24291760e-02 -9.11206603e-02
 -1.60467997e-01  1.44758029e-02 -2.47425772e-03  1.39562428e-01
  1.17651736e-02 -6.59413487e-02 -1.54661043e-02  3.82499620e-02
  1.01444446e-01  1.03162

In [60]:
test = np.load('test.npy')

In [1]:
import csv

with open('contigs.csv') as csvfile:
    data = list(csv.reader(csvfile, delimiter=","))
dna_sequences = [i[1] for i in data[1:]]

In [2]:
k = np.load('pretrained_4mer_embedding.npy')

NameError: name 'np' is not defined

In [7]:
from utils2 import calculate_tnf, calculate_dna2vec_embedding

In [ ]:
import numpy as np
import transformers
import torch
import torch.utils.data as util_data
import torch.nn as nn
import tqdm
import os
from typing import List, Tuple


def calculate_tnf(
    dna_sequences: List[List[str]], kernel: bool = False
) -> Tuple[np.ndarray, int]:
    """Calculates tetranucleotide frequencies in a list of DNA sequences.

    This function computes the frequencies of all possible tetranucleotides (sequences of four nucleotides)
    within each DNA sequence in `dna_sequences`. Corresponds to calculate 4-mers. There are 4^4=256 combinations.

    Optionally, a kernel transformation can be applied to the resulting frequency embeddings.

    Args:
        dna_sequences (List[List[str]]): A list of DNA sequences, where each sequence is a list of nucleotide characters (A, T, C, or G).
        kernel (bool, optional): If True, applies a kernel transformation to the calculated frequencies using a pre-defined kernel matrix. Defaults to False.

    Returns:
        Tuple[np.ndarray, int]: A tuple containing:
            - embedding (np.ndarray): A 2D numpy array of shape (n, 256), where `n` is the number of DNA sequences,
              and each row contains the tetranucleotide frequency vector for a sequence. If `kernel` is True,
              the frequencies are transformed by the kernel matrix.
            - count_N (int): Total number of tetranucleotides with nucleotide N across all dna_sequences.
    """

    nucleotides = ["A", "T", "C", "G"]
    tetra_nucleotides = [
        a + b + c + d
        for a in nucleotides
        for b in nucleotides
        for c in nucleotides
        for d in nucleotides
    ]

    # Build mapping from tetra-nucleotide to index
    tnf_index = {tn: i for i, tn in enumerate(tetra_nucleotides)}

    # Build embeddings by counting TNFs
    embedding = np.zeros((len(dna_sequences), len(tetra_nucleotides)))
    count_N = 0
    for j, seq in enumerate(dna_sequences):
        for i in range(len(seq) - 3):
            try:
                tetra_nuc = seq[i : i + 4]
                embedding[j, tnf_index[tetra_nuc]] += 1
            except KeyError:  # there exist nucleotide N which will give error
                count_N += 1

    # Convert counts to frequencies
    total_counts = np.sum(embedding, axis=1)
    embedding = embedding / total_counts[:, None]

    return embedding, count_N


def calculate_dna2vec_embedding(dna_sequences: List[List[str]]) -> np.array:
    """
    Calculates the DNA2Vec embedding for a list of DNA sequences.

    The function then multiplies the TNF embedding with a 4-mer embedding matrix to obtain
    the DNA2Vec embedding. The 4-mer embedding matrix is pretrained embeddings on the hg38 (human genome) obtained from https://github.com/MAGICS-LAB/DNABERT_S/blob/main/evaluate/helper/4mer_embedding.npy.
    See more in paper dna2vec https://arxiv.org/abs/1701.06279.

    Args:
        dna_sequences (List[List[str]]): A list of DNA sequences, where each sequence is a list
                                         of nucleotide characters (A, T, C, or G).
    Returns:
        np.ndarray: A 2D numpy array representing the DNA2Vec embedding, where each row corresponds
                    to the embedding of a DNA sequence.
    """

    tnf_embeddings = os.path.join("embeddings", "tnf.npy")
    if os.path.exists(tnf_embeddings):
        print(f"Load TNF-embedding from file {tnf_embeddings}")
        tnf_embedding = np.load(tnf_embeddings)
    else:
        tnf_embedding, _ = calculate_tnf(dna_sequences)

    pretrained_4mer_embedding = np.load(
        "pretrained_4mer_embedding.npy"
    )  # dim (256,100)

    print(type(tnf_embedding))
    print(pretrained_4mer_embedding.shape)
    embedding = np.dot(tnf_embedding, pretrained_4mer_embedding)

    return embedding


In [12]:
k = calculate_dna2vec_embedding(dna_sequences)

<class 'tuple'>
(256, 100)


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.